# Sinh dữ liệu Mixed Human–AI (Qwen2.5-7B & Llama-3.1-8B) trên Kaggle T4

**Hướng dẫn sử dụng (chỉ cần 3 bước):**
1. Ở cột bên phải mục **Input**, bấm **Upload -> New Dataset**, kéo file `prompts_to_generate.jsonl` (6 MB) vào, đặt tên bất kỳ (ví dụ `viet-ai-prompts`) rồi bấm **Create**.
2. Ở mục **Session options** bên phải: bật **Accelerator: `GPU T4 x2`** và bật **Internet: `On`**.
3. Bấm **Run All** (hoặc khuyên dùng: bấm nút **Save Version -> Save & Run All (Commit)** ở góc trên bên phải để Kaggle tự chạy ngầm trên đám mây kể cả khi bạn tắt máy; chạy xong vào tab **Output** tải file `generated_continuations.jsonl` về!).

In [ ]:
# Cell 1: Cài đặt vLLM và bitsandbytes bằng Python thuần (tránh lỗi cú pháp dấu ! trên Kaggle)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "vllm", "bitsandbytes"])
print("Đã cài đặt xong thư viện!")


In [ ]:
# Cell 2: Ghi file script kaggle_vllm_generate.py bằng Python thuần
from pathlib import Path
SCRIPT_CODE = 'from __future__ import annotations\n\nimport argparse\nimport gc\nimport glob\nimport json\nimport os\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Set\n\n# Tránh lỗi fork CUDA với vLLM trên Kaggle\nos.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")\n\nDEFAULT_MODEL_MAP = {\n    "Qwen/Qwen2.5-7B": "Qwen/Qwen2.5-7B",\n    "meta-llama/Llama-3.1-8B": "unsloth/Meta-Llama-3.1-8B",\n}\n\n\ndef find_input_file(explicit_path: str | None) -> Path:\n    if explicit_path and Path(explicit_path).exists():\n        return Path(explicit_path)\n    matches = glob.glob("/kaggle/input/**/prompts_to_generate.jsonl", recursive=True)\n    if matches:\n        print(f"[Auto-Detect] Đã tìm thấy file đầu vào tại: {matches[0]}", flush=True)\n        return Path(matches[0])\n    raise FileNotFoundError(\n        "Không tìm thấy prompts_to_generate.jsonl trong /kaggle/input/! "\n        "Hãy bấm Upload -> New Dataset ở thanh bên phải và tải file prompts_to_generate.jsonl lên."\n    )\n\n\ndef load_done_ids(output_path: Path) -> Set[str]:\n    done: Set[str] = set()\n    if output_path.exists():\n        with open(output_path, encoding="utf-8") as f:\n            for line in f:\n                line = line.strip()\n                if not line:\n                    continue\n                try:\n                    done.add(json.loads(line)["sample_id"])\n                except Exception:\n                    continue\n    return done\n\n\ndef run_with_vllm(\n    hf_repo: str,\n    items: List[Dict[str, Any]],\n    output_path: Path,\n    batch_size: int,\n    temperature: float,\n    top_p: float,\n) -> None:\n    import torch\n    from vllm import LLM, SamplingParams\n\n    num_gpus = torch.cuda.device_count()\n    if num_gpus == 0:\n        raise RuntimeError("Chưa bật GPU! Hãy chọn Accelerator -> GPU T4 x2 ở thanh bên phải.")\n\n    llm_kwargs: Dict[str, Any] = {\n        "model": hf_repo,\n        "dtype": "float16",\n        "max_model_len": 2048,\n        "gpu_memory_utilization": 0.90,\n        "trust_remote_code": True,\n    }\n    if num_gpus >= 2:\n        llm_kwargs["tensor_parallel_size"] = 2\n        print(f"[vLLM] Chạy {hf_repo} ở float16 trên 2 GPU (Tensor Parallel = 2)...", flush=True)\n    else:\n        llm_kwargs["quantization"] = "bitsandbytes"\n        llm_kwargs["load_format"] = "bitsandbytes"\n        print(f"[vLLM] Chỉ có 1 GPU -> Chạy {hf_repo} ở 4-bit bitsandbytes...", flush=True)\n\n    llm = LLM(**llm_kwargs)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n\n    for start in range(0, len(items), batch_size):\n        chunk = items[start : start + batch_size]\n        prompts = [x["prefix_text"] + " " for x in chunk]\n        params_list = [\n            SamplingParams(\n                temperature=temperature,\n                top_p=top_p,\n                repetition_penalty=1.08,\n                max_tokens=int(x.get("max_new_tokens", 380)),\n            )\n            for x in chunk\n        ]\n        outputs = llm.generate(prompts, params_list)\n        with open(output_path, "a", encoding="utf-8") as out_f:\n            for item, out in zip(chunk, outputs):\n                gen_text = out.outputs[0].text if out.outputs else ""\n                out_f.write(json.dumps({**item, "hf_repo_used": hf_repo, "raw_continuation": gen_text}, ensure_ascii=False) + "\\n")\n        print(f"[{hf_repo}] Đã lưu {min(start + batch_size, len(items))}/{len(items)} mẫu.", flush=True)\n\n    del llm\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\ndef run_with_transformers_fallback(\n    hf_repo: str,\n    items: List[Dict[str, Any]],\n    output_path: Path,\n    temperature: float,\n    top_p: float,\n) -> None:\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    print(f"[Transformers Fallback] Đang nạp {hf_repo} với device_map=\'auto\' (float16)...", flush=True)\n    tok = AutoTokenizer.from_pretrained(hf_repo, padding_side="left", trust_remote_code=True)\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    model = AutoModelForCausalLM.from_pretrained(\n        hf_repo,\n        torch_dtype=torch.float16,\n        device_map="auto",\n        trust_remote_code=True,\n    )\n    model.eval()\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    bs = 8\n    for start in range(0, len(items), bs):\n        chunk = items[start : start + bs]\n        prompts = [x["prefix_text"] + " " for x in chunk]\n        max_new = max(int(x.get("max_new_tokens", 380)) for x in chunk)\n        enc = tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(model.device)\n        in_len = enc.input_ids.shape[1]\n        with torch.no_grad():\n            gen_ids = model.generate(\n                **enc,\n                do_sample=True,\n                temperature=temperature,\n                top_p=top_p,\n                repetition_penalty=1.08,\n                max_new_tokens=max_new,\n                pad_token_id=tok.pad_token_id,\n            )\n        decoded = tok.batch_decode(gen_ids[:, in_len:], skip_special_tokens=True)\n        with open(output_path, "a", encoding="utf-8") as out_f:\n            for item, gen_text in zip(chunk, decoded):\n                out_f.write(json.dumps({**item, "hf_repo_used": hf_repo, "raw_continuation": gen_text}, ensure_ascii=False) + "\\n")\n        if (start // bs) % 5 == 0 or start + bs >= len(items):\n            print(f"[{hf_repo}] Đã lưu {min(start + bs, len(items))}/{len(items)} mẫu.", flush=True)\n\n    del model, tok\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input", default=None, help="Tự động tìm trong /kaggle/input nếu bỏ trống")\n    parser.add_argument("--output", default="/kaggle/working/generated_continuations.jsonl", type=Path)\n    parser.add_argument("--batch-size", type=int, default=200)\n    parser.add_argument("--temperature", type=float, default=0.7)\n    parser.add_argument("--top-p", type=float, default=0.9)\n    args = parser.parse_args()\n\n    in_path = find_input_file(args.input)\n    done_ids = load_done_ids(args.output)\n    print(f"Số mẫu đã hoàn thành từ trước: {len(done_ids)}", flush=True)\n\n    grouped: Dict[str, List[Dict[str, Any]]] = {}\n    with open(in_path, encoding="utf-8") as f:\n        for line in f:\n            obj = json.loads(line)\n            if obj["sample_id"] not in done_ids:\n                grouped.setdefault(obj["generator"], []).append(obj)\n\n    for gen_key, pending_items in grouped.items():\n        hf_repo = DEFAULT_MODEL_MAP.get(gen_key, gen_key)\n        print(f"=== Bắt đầu sinh {len(pending_items)} mẫu cho {gen_key} ({hf_repo}) ===", flush=True)\n        try:\n            run_with_vllm(hf_repo, pending_items, args.output, args.batch_size, args.temperature, args.top_p)\n        except Exception as exc:\n            print(f"[Cảnh báo] vLLM gặp lỗi ({exc}) -> Tự động chuyển sang Transformers...", flush=True)\n            done_now = load_done_ids(args.output)\n            rem = [x for x in pending_items if x["sample_id"] not in done_now]\n            if rem:\n                run_with_transformers_fallback(hf_repo, rem, args.output, args.temperature, args.top_p)\n\n    print("HOÀN TẤT 100%! Hãy tải file /kaggle/working/generated_continuations.jsonl về máy.", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
Path("kaggle_vllm_generate.py").write_text(SCRIPT_CODE, encoding="utf-8")
print("Đã tạo xong kaggle_vllm_generate.py!")


In [ ]:
# Cell 3: Chạy tiến trình sinh toàn bộ 2,650 mẫu
import subprocess, sys
subprocess.run([sys.executable, "kaggle_vllm_generate.py"], check=True)


In [ ]:
# Cell 4: Kiểm tra kết quả đầu ra
from pathlib import Path
out = Path("/kaggle/working/generated_continuations.jsonl")
lines = out.read_text(encoding="utf-8").splitlines()
print(f"Tổng số dòng đã sinh: {len(lines)}")
print("Mẫu đầu tiên:", lines[0][:300] if lines else "Rỗng")
